# Hallucination detection

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/hallucination-detection). Compare local NLI checks against a hosted groundedness judge on Ridgeline Outfitters policy answers.

## Install

Use a fresh Python 3.12 notebook kernel. The first local NLI check downloads model weights; a warning about word overlap means you are using the fallback instead.

In [ ]:
import subprocess
import sys

if sys.platform.startswith("linux"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "torch>=2.10,<3", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit[nli]==0.3.7", "sentencepiece"], check=True);

## Set your Future AGI credentials

Existing environment variables are used first. Otherwise enter each value at the hidden prompt.

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    value = getpass(f"{name}: ")
    if not value.strip():
        raise ValueError(f"{name} is required")
    os.environ[name] = value.strip()

for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    load_secret(name)

## Score a faithful response

In [ ]:
from fi.evals import evaluate

context = (
    "Pro plan subscriptions can be canceled anytime from Account Settings > Billing. "
    "Cancellation takes effect at the end of the current billing cycle, and no partial "
    "refunds are issued for unused time. Annual plans can be downgraded to monthly "
    "only at renewal, not mid-cycle."
)

question = "How do I cancel my Pro plan, and will I get a refund for the unused time?"

# A response that faithfully reflects the context
response = (
    "You can cancel your Pro plan anytime from Account Settings > Billing. "
    "Cancellation takes effect at the end of your current billing cycle, and you won't "
    "get a refund for the unused time."
)

result = evaluate(
    "faithfulness",
    output=response,
    context=context,
    input=question,
)

print(f"Faithfulness score : {result.score:.2f}")
print(f"Passed             : {result.passed}")
print(f"Reason             : {result.reason}")

## Detect a hallucinated response

In [ ]:
from fi.evals import evaluate

hallucinated_response = (
    "You can cancel your Pro plan anytime, and it takes effect immediately. "
    "You'll receive a prorated refund for the unused portion of the billing cycle."
)

result = evaluate(
    "faithfulness",
    output=hallucinated_response,
    context=context,
    input=question,
)

print(f"Faithfulness score : {result.score:.2f}")
print(f"Passed             : {result.passed}")
print(f"Reason             : {result.reason}")

## Check groundedness on an ungrounded response

In [ ]:
from fi.evals import evaluate

# A response that adds a fact not present in the context
ungrounded_response = (
    "You can cancel your Pro plan anytime from Account Settings > Billing. "
    "Cancellation takes effect at the end of your current billing cycle, with no refund "
    "for unused time. Canceling also removes you from the referral rewards program."
)

result = evaluate(
    "groundedness",
    output=ungrounded_response,
    context=context,
    input=question,
    model="turing_small",
)

if result.passed is None:
    raise RuntimeError(result.error or result.reason or "No groundedness verdict")
print(f"Passed : {result.passed}")
print(f"Reason : {result.reason}")
unsupported_id = result.metadata["eval_id"]

## Check groundedness on a clean response

In [ ]:
from fi.evals import evaluate

clean_response = (
    "You can cancel your Pro plan anytime from Account Settings > Billing. "
    "Cancellation takes effect at the end of your current billing cycle, and there's no "
    "refund for unused time."
)

result = evaluate(
    "groundedness",
    output=clean_response,
    context=context,
    input=question,
    model="turing_small",
)

if result.passed is None:
    raise RuntimeError(result.error or result.reason or "No groundedness verdict")
print(f"Passed : {result.passed}")
print(f"Reason : {result.reason}")
# Keep the hosted IDs so you can retrieve the original verdicts later.
from pathlib import Path
Path("groundedness-ids.txt").write_text(f"{unsupported_id}\n{result.metadata['eval_id']}\n")

## Combine both metrics in one evaluate() call

In [ ]:
from fi.evals import evaluate

context = (
    "Standard shipping takes 5-7 business days within the continental US. Expedited "
    "shipping is available for an additional $12.99 and delivers in 2-3 business days. "
    "International shipping is not currently supported. Orders ship Monday through "
    "Friday, excluding federal holidays."
)

question = "What shipping options are available and how long does each take?"

response = (
    "Standard shipping takes 5-7 business days within the continental US. For an "
    "additional $12.99, expedited shipping delivers in 2-3 business days. Orders ship "
    "Monday through Friday, excluding federal holidays."
)

results = evaluate(
    ["faithfulness", "groundedness"],
    output=response,
    context=context,
    input=question,
)

# Iterate over both results
for result in results:
    status = "PASS" if result.passed else "FAIL"
    print(f"{result.eval_name:<15} score={result.score:.2f}  {status}")
    print(f"  Reason: {result.reason}")
    print()

# Or look up by name directly
faith_result = results.get("faithfulness")
ground_result = results.get("groundedness")

print(f"Both metrics passed: {faith_result.passed and ground_result.passed}")

Inspect the reasons alongside the policy text. A missed claim or low local score is not proof that an answer is false.